# 02 — Contextual Grounding with Bedrock Managed Knowledge Bases

This notebook demonstrates how to use **Bedrock Guardrails contextual grounding** with a Managed Knowledge Base to detect and block hallucinated or irrelevant responses.

### What is contextual grounding?

Contextual grounding checks whether a model's response is:
1. **Grounded** — factually supported by the retrieved source chunks
2. **Relevant** — answers the user's actual question

If the response fails either check (below your configured threshold), it's **blocked**.

### How it works with Managed KBs

```
User Query
    │
    ▼
AgenticRetrieveStream (with policyConfiguration)
    │
    ├── 1. Retrieve chunks from Managed KB
    ├── 2. Generate response using FM
    └── 3. Guardrail contextual grounding check:
           • Is response grounded in retrieved chunks? (threshold: 0.7)
           • Is response relevant to user query? (threshold: 0.7)
           • If PASS → return response
           • If FAIL → BLOCK response
```

### Key difference from DIY KBs

| | DIY KB | Managed KB |
|---|---|---|
| API | `RetrieveAndGenerate` | `AgenticRetrieveStream` |
| Parameter | `guardrailConfiguration` | `policyConfiguration` |
| Actions | BLOCK + MASK | **BLOCK only** |

### How this differs from AgentCore Policy Guardrails (Notebook 01)

| | Notebook 01 (AgentCore Policy) | This Notebook (Contextual Grounding) |
|---|---|---|
| Layer | Gateway perimeter | KB retrieval response |
| Sees retrieved chunks? | No (only sees final input/output) | **Yes** (compares response to source chunks) |
| Purpose | Block harmful inputs before they reach the agent | Block hallucinated outputs after generation |
| Integration | Cedar-style policies via AgentCore Gateway | `policyConfiguration` in AgenticRetrieveStream |
| Categories | Content filter, prompt attack, PII | **Grounding + Relevance** |

### Prerequisites

- AWS credentials with Bedrock and IAM permissions
- Model access enabled for embedding and generation models
- Region: us-east-1

### Reference

- [Contextual grounding check](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-contextual-grounding-check.html)
- [AgenticRetrieveStream API](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_agent-runtime_AgenticRetrieveStream.html)
- [Create a guardrail](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-create.html)

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --quiet

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Step 1 — Configuration

*Set up AWS clients, generate unique resource names, and configure the generation model.*

In [ ]:
import boto3
import sys
import time
import os
import json
import logging
import pprint

try:
    from dotenv import load_dotenv; load_dotenv('../.env')
except ImportError:
    pass

sys.path.insert(0, "..")

region = 'us-east-1'
os.environ['AWS_REGION'] = region
os.environ['AWS_DEFAULT_REGION'] = region

sts_client = boto3.client('sts', region_name=region)
s3_client = boto3.client('s3', region_name=region)
bedrock_client = boto3.client('bedrock', region_name=region)
runtime_client = boto3.client('bedrock-agent-runtime', region_name=region)
account_id = sts_client.get_caller_identity()['Account']

logging.basicConfig(
    format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

knowledge_base_name = f'bmkb-grounding-{suffix}'
bucket_name = f'bedrock-bmkb-grounding-{suffix}-{account_id}'

# Generation model (CRIS inference profile)
region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

embedding_model = None  # Let ManagedKnowledgeBase pick the default

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:     {region}')
print(f'Account:    {account_id}')
print(f'KB Name:    {knowledge_base_name}')
print(f'Bucket:     {bucket_name}')
print(f'Gen Model:  {generation_model_arn}')

## Step 2 — Create S3 Bucket and Upload Documents

*Upload the Octank Financial 10K report — this is the grounding source. Contextual grounding will verify that model responses are supported by the content in this document.*

In [ ]:
try:
    s3_client.head_bucket(Bucket=bucket_name)
    print(f'Bucket {bucket_name} already exists')
except Exception:
    print(f'Creating bucket {bucket_name}')
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': region}
        )

file_to_upload = '../synthetic_dataset/octank_financial_10K.pdf'
print(f'Uploading {file_to_upload} to {bucket_name}')
s3_client.upload_file(file_to_upload, bucket_name, 'octank_financial_10K.pdf')
print('Document uploaded successfully.')

## Step 3 — Create the Bedrock Managed Knowledge Base

*Uses the shared utility for full lifecycle: IAM role, KB creation with Bedrock-managed vector store, and S3 data source.*

In [ ]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

kb = ManagedKnowledgeBase(
    kb_name=knowledge_base_name,
    bucket_name=bucket_name,
    embedding_model=embedding_model,
    enable_logging=True,
    region_name=region,
    suffix=suffix,
)

print(f'\nKB ID: {kb.kb_id}')
print(f'DS ID: {kb.ds_id}')

kb_id = kb.kb_id
%store kb_id

## Step 4 — Ingest Documents

*Bedrock crawls the S3 bucket, parses documents, generates embeddings, and indexes them. The 30s wait ensures the KB is ready to accept the ingestion job.*

In [ ]:
time.sleep(30)
job = kb.start_ingestion_job()

## Step 5 — Create a Guardrail with Contextual Grounding

*Create a Bedrock Guardrail that checks:*
- **Grounding threshold: 0.7** — at least 70% of the response must be supported by retrieved source chunks
- **Relevance threshold: 0.7** — response must be at least 70% relevant to the user's query

*If either check fails → response is BLOCKED.*

In [ ]:
guardrail_response = bedrock_client.create_guardrail(
    name=f'bmkb-grounding-guardrail-{suffix}',
    description='Contextual grounding check for Managed KB responses',
    contextualGroundingPolicyConfig={
        'filtersConfig': [
            {
                'type': 'GROUNDING',
                'threshold': 0.7,
                'action': 'BLOCK',
            },
            {
                'type': 'RELEVANCE',
                'threshold': 0.7,
                'action': 'BLOCK',
            },
        ]
    },
    blockedInputMessaging='Your query could not be processed.',
    blockedOutputsMessaging='The response was blocked because it could not be verified against the source documents.',
)

guardrail_id = guardrail_response['guardrailId']
guardrail_version = guardrail_response['version']  # 'DRAFT'

print(f'Guardrail ID:      {guardrail_id}')
print(f'Guardrail Version: {guardrail_version}')
print(f'\nContextual grounding configured:')
print(f'  Grounding threshold: 0.7 (BLOCK if response is not grounded in sources)')
print(f'  Relevance threshold: 0.7 (BLOCK if response is not relevant to query)')

## Step 6 — Query WITHOUT Guardrail (Baseline)

*Query the KB normally to see the response without grounding checks. This establishes a baseline for comparison.*

In [ ]:
query = "What is Octank Financial's total revenue and growth rate?"

response = runtime_client.agentic_retrieve_stream(
    messages=[{'role': 'user', 'content': {'text': query}}],
    retrievers=[{
        'configuration': {
            'knowledgeBase': {
                'knowledgeBaseId': kb_id,
                'retrievalOverrides': {'maxNumberOfResults': 5},
            }
        }
    }],
    agenticRetrieveConfiguration={
        'foundationModelConfiguration': {
            'bedrockFoundationModelConfiguration': {
                'modelConfiguration': {'modelArn': generation_model_arn}
            },
            'type': 'BEDROCK_FOUNDATION_MODEL',
        },
        'foundationModelType': 'CUSTOM',
        'maxAgentIteration': 3,
        'rerankingModelType': 'MANAGED',
    },
    generateResponse=True,
)

gen_resp = None
for event in response['stream']:
    if 'result' in event:
        gen_resp = event['result'].get('generatedResponse')

print('=== Without Guardrail (baseline) ===')
if gen_resp:
    print(f'Answer: {gen_resp["answer"][:500]}')
    print(f'\nCitations: {len(gen_resp.get("citations", []))}')
else:
    print('No generated response')

## Step 7 — Query WITH Contextual Grounding Guardrail (Grounded Query)

*Same query, but now with `policyConfiguration` enabled. The guardrail checks if the generated response is factually grounded in the retrieved chunks. Since this query is about content IN the KB, it should PASS.*

In [ ]:
response = runtime_client.agentic_retrieve_stream(
    messages=[{'role': 'user', 'content': {'text': query}}],
    retrievers=[{
        'configuration': {
            'knowledgeBase': {
                'knowledgeBaseId': kb_id,
                'retrievalOverrides': {'maxNumberOfResults': 5},
            }
        }
    }],
    agenticRetrieveConfiguration={
        'foundationModelConfiguration': {
            'bedrockFoundationModelConfiguration': {
                'modelConfiguration': {'modelArn': generation_model_arn}
            },
            'type': 'BEDROCK_FOUNDATION_MODEL',
        },
        'foundationModelType': 'CUSTOM',
        'maxAgentIteration': 3,
        'rerankingModelType': 'MANAGED',
    },
    generateResponse=True,
    policyConfiguration={
        'bedrockGuardrailConfiguration': {
            'guardrailId': guardrail_id,
            'guardrailVersion': guardrail_version,
        }
    },
)

gen_resp = None
results = []
traces = []
for event in response['stream']:
    if 'traceEvent' in event:
        traces.append(event['traceEvent'])
    elif 'result' in event:
        results = event['result'].get('results', [])
        gen_resp = event['result'].get('generatedResponse')

print('=== With Contextual Grounding Guardrail ===')
print(f'Guardrail: {guardrail_id} (grounding=0.7, relevance=0.7)')
print(f'Chunks retrieved: {len(results)}')
print()

if gen_resp:
    print('RESULT: PASSED grounding check')
    print(f'Answer: {gen_resp["answer"][:500]}')
    print(f'\nCitations: {len(gen_resp.get("citations", []))}')
else:
    print('RESULT: BLOCKED by guardrail (ungrounded or irrelevant)')
    print('The model response was not sufficiently grounded in the retrieved sources.')

# Show guardrail trace if present
guardrail_traces = [t for t in traces if 'guardrail' in json.dumps(t).lower()]
if guardrail_traces:
    print(f'\n--- Guardrail Trace ({len(guardrail_traces)} events) ---')
    for t in guardrail_traces[:3]:
        print(json.dumps(t, indent=2, default=str)[:500])

### Diagnostic: Inspect Trace Events

*Trace events show what happened inside the pipeline — including whether the guardrail evaluated the response. If you see guardrail-related traces, it confirms the contextual grounding check ran and had access to the retrieved chunks.*

In [ ]:
print(f'Total trace events captured: {len(traces)}\n')

for i, trace in enumerate(traces):
    attrs = trace.get('attributes', {})
    step = attrs.get('step', 'unknown')
    status = attrs.get('status', '')
    message = attrs.get('message', '')
    
    # Summarize each trace step
    summary = f'  Step {i+1}: [{step}]'
    if status:
        summary += f' status={status}'
    if message:
        summary += f' — {message[:120]}'
    print(summary)

    # Highlight guardrail-specific information
    trace_str = json.dumps(trace, default=str).lower()
    if any(kw in trace_str for kw in ['guardrail', 'grounding', 'relevance', 'policy']):
        print(f'         ^ Guardrail evaluation detected in this trace')
        print(f'         Raw: {json.dumps(attrs, indent=2, default=str)[:600]}')

if not traces:
    print('No trace events captured. Ensure generateResponse=True is set.')
    print('If policyConfiguration was rejected, traces may not include guardrail steps.')

## Step 8 — Test with an Ungrounded Query

*Ask something NOT in the KB. If the model hallucinates an answer instead of saying "I don't know", the guardrail should catch the ungrounded content and block it.*

In [ ]:
ungrounded_query = "What is the weather forecast for Tokyo next week?"

response = runtime_client.agentic_retrieve_stream(
    messages=[{'role': 'user', 'content': {'text': ungrounded_query}}],
    retrievers=[{
        'configuration': {
            'knowledgeBase': {
                'knowledgeBaseId': kb_id,
                'retrievalOverrides': {'maxNumberOfResults': 5},
            }
        }
    }],
    agenticRetrieveConfiguration={
        'foundationModelConfiguration': {
            'bedrockFoundationModelConfiguration': {
                'modelConfiguration': {'modelArn': generation_model_arn}
            },
            'type': 'BEDROCK_FOUNDATION_MODEL',
        },
        'foundationModelType': 'CUSTOM',
        'maxAgentIteration': 3,
        'rerankingModelType': 'MANAGED',
    },
    generateResponse=True,
    policyConfiguration={
        'bedrockGuardrailConfiguration': {
            'guardrailId': guardrail_id,
            'guardrailVersion': guardrail_version,
        }
    },
)

gen_resp = None
for event in response['stream']:
    if 'result' in event:
        gen_resp = event['result'].get('generatedResponse')

print(f'Query: "{ungrounded_query}"')
print(f'(This topic is NOT in the KB)\n')

if gen_resp:
    answer = gen_resp['answer']
    if any(phrase in answer.lower() for phrase in ["i don't know", "not found", "no information", "cannot find", "not available"]):
        print('RESULT: PASSED (model correctly said it does not know)')
    else:
        print('RESULT: PASSED (model returned a response — guardrail deemed it grounded)')
    print(f'Answer: {answer[:300]}')
else:
    print('RESULT: BLOCKED — guardrail detected ungrounded/irrelevant content')
    print('This is the expected behavior when the model tries to answer outside the KB knowledge.')

## Step 9 — Test with a Partially Grounded Query

*Ask a question that mixes KB content with speculation. This tests whether the guardrail can detect when a response goes beyond what the sources support.*

In [ ]:
speculative_query = "Based on Octank's financials, predict their stock price in 2030 and which competitors they will acquire."

response = runtime_client.agentic_retrieve_stream(
    messages=[{'role': 'user', 'content': {'text': speculative_query}}],
    retrievers=[{
        'configuration': {
            'knowledgeBase': {
                'knowledgeBaseId': kb_id,
                'retrievalOverrides': {'maxNumberOfResults': 5},
            }
        }
    }],
    agenticRetrieveConfiguration={
        'foundationModelConfiguration': {
            'bedrockFoundationModelConfiguration': {
                'modelConfiguration': {'modelArn': generation_model_arn}
            },
            'type': 'BEDROCK_FOUNDATION_MODEL',
        },
        'foundationModelType': 'CUSTOM',
        'maxAgentIteration': 3,
        'rerankingModelType': 'MANAGED',
    },
    generateResponse=True,
    policyConfiguration={
        'bedrockGuardrailConfiguration': {
            'guardrailId': guardrail_id,
            'guardrailVersion': guardrail_version,
        }
    },
)

gen_resp = None
for event in response['stream']:
    if 'result' in event:
        gen_resp = event['result'].get('generatedResponse')

print(f'Query: "{speculative_query}"')
print(f'(Asks for speculation beyond what the document contains)\n')

if gen_resp:
    print('RESULT: PASSED (model response was deemed grounded)')
    print(f'Answer: {gen_resp["answer"][:300]}')
else:
    print('RESULT: BLOCKED — guardrail detected speculative/ungrounded content')
    print('The model tried to speculate beyond what the source documents support.')

## Step 10 — Adjust Grounding Thresholds

*Create a stricter guardrail (0.9 threshold) and compare behavior. Higher thresholds are more aggressive at blocking — useful for compliance-critical applications.*

| Threshold | Behavior | Use Case |
|-----------|----------|----------|
| 0.5 | Permissive — allows partially grounded responses | Casual Q&A |
| 0.7 | Balanced — good default | Business reports |
| 0.9 | Strict — blocks unless almost perfectly grounded | Legal/compliance |
| 0.95 | Very strict — blocks any extrapolation | Medical/safety-critical |

In [ ]:
strict_response = bedrock_client.create_guardrail(
    name=f'bmkb-strict-grounding-{suffix}',
    description='Strict contextual grounding (0.9 threshold)',
    contextualGroundingPolicyConfig={
        'filtersConfig': [
            {'type': 'GROUNDING', 'threshold': 0.9, 'action': 'BLOCK'},
            {'type': 'RELEVANCE', 'threshold': 0.9, 'action': 'BLOCK'},
        ]
    },
    blockedInputMessaging='Query blocked.',
    blockedOutputsMessaging='Response blocked: insufficient grounding in source documents.',
)

strict_guardrail_id = strict_response['guardrailId']
print(f'Strict guardrail created: {strict_guardrail_id} (threshold=0.9)')

In [ ]:
# Test the same grounded query with the strict threshold
response = runtime_client.agentic_retrieve_stream(
    messages=[{'role': 'user', 'content': {'text': query}}],
    retrievers=[{
        'configuration': {
            'knowledgeBase': {
                'knowledgeBaseId': kb_id,
                'retrievalOverrides': {'maxNumberOfResults': 5},
            }
        }
    }],
    agenticRetrieveConfiguration={
        'foundationModelConfiguration': {
            'bedrockFoundationModelConfiguration': {
                'modelConfiguration': {'modelArn': generation_model_arn}
            },
            'type': 'BEDROCK_FOUNDATION_MODEL',
        },
        'foundationModelType': 'CUSTOM',
        'maxAgentIteration': 3,
        'rerankingModelType': 'MANAGED',
    },
    generateResponse=True,
    policyConfiguration={
        'bedrockGuardrailConfiguration': {
            'guardrailId': strict_guardrail_id,
            'guardrailVersion': 'DRAFT',
        }
    },
)

gen_resp = None
for event in response['stream']:
    if 'result' in event:
        gen_resp = event['result'].get('generatedResponse')

print(f'Query: "{query}"')
print(f'Guardrail: strict (threshold=0.9)\n')

if gen_resp:
    print('RESULT: PASSED strict grounding check')
    print(f'Answer: {gen_resp["answer"][:300]}')
else:
    print('RESULT: BLOCKED by strict guardrail')
    print('Response did not meet the 90% grounding threshold — even though the same')
    print('query passed with 0.7 threshold. This shows the trade-off between')
    print('hallucination prevention and response availability.')

## Step 11 — Compare Results Side-by-Side

*Run all test queries with both thresholds to see the practical impact of tuning.*

In [ ]:
def test_with_guardrail(test_query, gr_id, gr_version, label):
    """Run a query with a guardrail and return pass/block status."""
    resp = runtime_client.agentic_retrieve_stream(
        messages=[{'role': 'user', 'content': {'text': test_query}}],
        retrievers=[{
            'configuration': {
                'knowledgeBase': {
                    'knowledgeBaseId': kb_id,
                    'retrievalOverrides': {'maxNumberOfResults': 5},
                }
            }
        }],
        agenticRetrieveConfiguration={
            'foundationModelConfiguration': {
                'bedrockFoundationModelConfiguration': {
                    'modelConfiguration': {'modelArn': generation_model_arn}
                },
                'type': 'BEDROCK_FOUNDATION_MODEL',
            },
            'foundationModelType': 'CUSTOM',
            'maxAgentIteration': 3,
            'rerankingModelType': 'MANAGED',
        },
        generateResponse=True,
        policyConfiguration={
            'bedrockGuardrailConfiguration': {
                'guardrailId': gr_id,
                'guardrailVersion': gr_version,
            }
        },
    )
    gen_resp = None
    for event in resp['stream']:
        if 'result' in event:
            gen_resp = event['result'].get('generatedResponse')
    return 'PASS' if gen_resp else 'BLOCK'


test_queries = [
    ("What is Octank Financial's total revenue?", "Grounded (in KB)"),
    ("What are the main risk factors in the report?", "Grounded (in KB)"),
    ("What is the weather forecast for Tokyo next week?", "Ungrounded (not in KB)"),
    ("Predict Octank's stock price in 2030.", "Speculative (beyond KB)"),
]

print(f'{"Query":<55} {"Threshold=0.7":<15} {"Threshold=0.9":<15}')
print('=' * 85)

for q, desc in test_queries:
    r1 = test_with_guardrail(q, guardrail_id, guardrail_version, '0.7')
    r2 = test_with_guardrail(q, strict_guardrail_id, 'DRAFT', '0.9')
    print(f'{desc:<55} {r1:<15} {r2:<15}')

## Step 12 — Cleanup

*Delete guardrails and KB resources. Uncomment the cells below to run cleanup.*

In [ ]:
# Delete guardrails
print('Deleting guardrails...')
bedrock_client.delete_guardrail(guardrailIdentifier=guardrail_id)
print(f'  Deleted: {guardrail_id}')
bedrock_client.delete_guardrail(guardrailIdentifier=strict_guardrail_id)
print(f'  Deleted: {strict_guardrail_id}')
print('Guardrails deleted.')

In [ ]:
# Delete Knowledge Base and S3 bucket
# kb.delete_kb(delete_s3_bucket=True)
# print('KB and bucket deleted.')

## Summary

### Integration pattern

```python
# Add to any AgenticRetrieveStream call:
policyConfiguration={
    'bedrockGuardrailConfiguration': {
        'guardrailId': 'your-guardrail-id',
        'guardrailVersion': 'DRAFT',  # or published version number
    }
}
```

### How the grounding check works

| Step | What happens |
|------|-------------|
| 1 | Managed KB retrieves relevant chunks |
| 2 | FM generates a response using the chunks |
| 3 | Guardrail checks grounding (response vs chunks) |
| 4 | Guardrail checks relevance (response vs query) |
| 5 | If PASS → return response; if FAIL → BLOCK |

### Complementary guardrail layers

| Layer | Tool | What it catches |
|-------|------|----------------|
| **Perimeter** (input) | AgentCore Policy (Notebook 01) | Harmful prompts, PII, prompt injection |
| **Response** (output) | Contextual Grounding (this notebook) | Hallucinations, irrelevant answers |

These two layers are complementary — use both for defense-in-depth.

### Threshold guide

| Use case | Grounding | Relevance |
|----------|-----------|----------|
| Casual Q&A | 0.5 | 0.5 |
| Business reports | 0.7 | 0.7 |
| Legal/compliance | 0.9 | 0.9 |
| Medical/safety-critical | 0.95 | 0.95 |

### Limitations with Managed KBs

- Only `BLOCK` action supported (not `MASK`) for AgenticRetrieveStream
- Grounding source limit: 100,000 characters
- Query limit: 1,000 characters
- Response limit: 5,000 characters

### Documentation

- [Contextual grounding check](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-contextual-grounding-check.html)
- [AgenticRetrieveStream API Reference](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_agent-runtime_AgenticRetrieveStream.html)
- [Create a guardrail](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-create.html)